In [1]:
from ast import literal_eval
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer
import warnings


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
# Load the train dataset
dataset = pd.read_csv("train.csv")

# Flatten the JSON dataset
records = []
for _, row in dataset.iterrows():
    problems = literal_eval(row["problems"])
    record = {
        "id": row["id"],
        "paragraph": row["paragraph"],
        "question": problems["question"],
        "choices": problems["choices"],
        "answer": problems.get("answer", None),
        "question_plus": problems.get("question_plus", None),
    }
    # Include 'question_plus' if it exists
    if "question_plus" in problems:
        record["question_plus"] = problems["question_plus"]
    records.append(record)

# Convert to DataFrame
df = pd.DataFrame(records)

### Token

In [ ]:
# Tokenizer 로드
print("Tokenizer 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained("beomi/gemma-ko-2b")
print("✓ Tokenizer 로드 완료!")

Tokenizer 로딩 중...
✓ Tokenizer 로드 완료!


In [ ]:
print("Paragraph Token 분석 시작")
print("=" * 50)

# Token 수 계산
print("\nToken 수 계산 중... (시간이 걸릴 수 있습니다)")
df["token_count"] = df["paragraph"].apply(lambda x: len(tokenizer.encode(x, add_special_tokens=True)))
print("✓ Token 수 계산 완료!")

Paragraph Token 분석 시작

Token 수 계산 중... (시간이 걸릴 수 있습니다)
✓ Token 수 계산 완료!


In [ ]:
# ===== 기본 통계 =====
print("\n" + "=" * 50)
print("1. 기본 통계")
print("=" * 50)
print(f"전체 데이터 개수: {len(df)}")
print(f"평균 Token 수: {df['token_count'].mean():.2f}")
print(f"중앙값 Token 수: {df['token_count'].median():.2f}")
print(f"표준편차: {df['token_count'].std():.2f}")
print(f"최소 Token 수: {df['token_count'].min()}")
print(f"최대 Token 수: {df['token_count'].max()}")


1. 기본 통계
전체 데이터 개수: 2031
평균 Token 수: 531.22
중앙값 Token 수: 508.00
표준편차: 340.73
최소 Token 수: 11
최대 Token 수: 1532


In [ ]:
# ===== Token 범위별 분포 =====
print("\n" + "=" * 50)
print("3. Token 범위별 분포")
print("=" * 50)
bins = [0, 100, 200, 300, 400, 500, 1000, 2000, float("inf")]
labels = ["0-100", "100-200", "200-300", "300-400", "400-500", "500-1000", "1000-2000", "2000+"]
df["token_range"] = pd.cut(df["token_count"], bins=bins, labels=labels)

range_counts = df["token_range"].value_counts().sort_index()
for range_label, count in range_counts.items():
    pct = count / len(df) * 100
    print(f"{range_label} tokens: {count}개 ({pct:.2f}%)")


3. Token 범위별 분포
0-100 tokens: 295개 (14.52%)
100-200 tokens: 109개 (5.37%)
200-300 tokens: 131개 (6.45%)
300-400 tokens: 184개 (9.06%)
400-500 tokens: 274개 (13.49%)
500-1000 tokens: 803개 (39.54%)
1000-2000 tokens: 235개 (11.57%)
2000+ tokens: 0개 (0.00%)
